## Installtion

In [ ]:
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"


## import

In [ ]:
from google.colab import drive
import open_dataset_store as ds
import numpy as np
import pandas as pd
import plotly.express as px
import sims as kf_module
from scipy.spatial.transform import Rotation as R

In [ ]:
print('Open Dataset Store Version :',ds.__version__)
drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = ds.quick_start(dataset_path, backend='local')
store.summary('experiments')

In [ ]:
store.list_entities('subjects')

In [ ]:
store.list_entries('experiments')
# store.get_entry("experiments", "experiments_0001")

In [ ]:
df_1 = store.get_entry_raw_data("experiments", "experiments_0001")
df_2 = store.get_entry_raw_data("experiments", "experiments_0003")

In [ ]:
def rotmat_to_quat(R_mat):
    """Convert 3x3 rotation matrix to quaternion [w, x, y, z]"""
    rot = R.from_matrix(R_mat)
    q = rot.as_quat()  # scipy returns [x, y, z, w]
    return np.array([q[3], q[0], q[1], q[2]])  # [w, x, y, z]
    
def DEKF(df,device_prefix):
    accel_cols = [f'{device_prefix}_AX', f'{device_prefix}_AY', f'{device_prefix}_AZ']
    gyro_cols = [f'{device_prefix}_GX', f'{device_prefix}_GY', f'{device_prefix}_GZ']

    accel = df[accel_cols].to_numpy(dtype=float)
    gyro  = df[gyro_cols].to_numpy(dtype=float)
    
    # Normalize accelerometer → unit vector (gravity direction)
    accel_norm = np.linalg.norm(accel, axis=1, keepdims=True)
    accel_norm = np.where(accel_norm < 1e-8, 1.0, accel_norm)
    y_data = accel / accel_norm

    # Filter Configuration
    E = np.array([[0.0, 0.0, -1.0]])
    dt = 0.01
    # Tuning parameters
    Sigma_gyro = (0.02**2) * np.eye(3)   # Gyro noise
    Sigma_bias = (1e-5**2) * np.eye(3)   # Gyro bias random walk
    Sigma_q = np.block([
        [Sigma_gyro, np.zeros((3, 3))],
        [np.zeros((3, 3)), Sigma_bias],
    ])
    Sigma_m = (0.15**2) * np.eye(3)

    kf = kf_module.LinearKF(use_joseph=True, symmetrize=True)
    iekf = kf_module.SO3IMUSensorFusionBiasEKF(kf=kf, inertial_directions=E, dt=dt)
    res = iekf.run_filter(
        R0_plus=np.eye(3),
        b0_plus=np.zeros(3),
        P0=np.diag([1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4]),
        Omegas=gyro,
        Ys=y_data,
        Sigma_q=Sigma_q,
        Sigma_m=Sigma_m
    )
    
    R_est = res["R_plus"]  # (T, 3, 3)
    # Convert to quaternions
    quats = np.array([rotmat_to_quat(R_mat) for R_mat in R_est])
    
    # Output DataFrame
    out = pd.DataFrame()
    if 'Timestamp' in df.columns:
        out['Timestamp'] = df['Timestamp'].values
    else:
        out['Timestamp'] = np.arange(len(df))
    
    out[f'{device_prefix}_qw'] = quats[:, 0]
    out[f'{device_prefix}_qx'] = quats[:, 1]
    out[f'{device_prefix}_qy'] = quats[:, 2]
    out[f'{device_prefix}_qz'] = quats[:, 3]
    
    return out

def Orientation_Estimation(df, device_list=None):
    """
    Run DEKF on multiple devices and combine results.
    """
    if device_list is None:
        # Auto-detect devices from _AX columns
        accel_cols = [col for col in df.columns if col.endswith('_AX')]
        device_list = sorted(set(col.replace('_AX', '') for col in accel_cols))
    
    results = []
    for dev in device_list:
        print(f"Processing {dev} ...")
        try:
            res = DEKF(df, dev)
            results.append(res)
        except Exception as e:
            print(f"    Error on {dev}: {e}")
    
    if not results:
        raise ValueError("No devices processed successfully.")
    
    # Merge all
    final = results[0]
    for r in results[1:]:
        if 'Timestamp' in r.columns:
            final = pd.merge(final, r, on='Timestamp', how='outer')
        else:
            final = pd.concat([final, r], axis=1)
    
    if 'Timestamp' in final.columns:
        final = final.sort_values('Timestamp').reset_index(drop=True)
    
    return final

In [ ]:
# full_df = Orientation_Estimation(df_1)

In [ ]:
df_1.head()

In [ ]:
fig = px.line(
    df_1,
    x='Timestamp',
    width=1500,
    y=['device1_IMU1_GX','device1_IMU1_GY','device1_IMU1_GZ'],
    title='Device 1 IMU Accelerometer Data Over Time'
)

fig.show(renderer="colab+vscode")

In [123]:
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

def segmentation(df, window=120, config=('device2_IMU1', 'device4_IMU1','device7_IMU1', 'device6_IMU1'), congif_1=('GX','GY', 'GZ')):
    gyro_cols = [col for col in df.columns if col.startswith(config) and col.endswith(congif_1)]
    df['total_gyro'] = df[gyro_cols].abs().sum(axis=1)
    df['moving_avg'] = df['total_gyro'].rolling(window=window, min_periods=1).mean()

    # 1. Identify 5 Main Reps
    main_valleys, _ = find_peaks(-df['moving_avg'], height=-100, distance=1500)
    
    fig = px.line(df, x='Timestamp', y=gyro_cols, width=1500, title='Strict 3-Stage Segmentation')
    fig.update_traces(visible='legendonly')
    fig.add_scatter(x=df['Timestamp'], y=df['moving_avg'], mode='lines', name='Moving Avg', line=dict(color='red', width=3))

    # 2. Iterate through each rep window
    for i in range(len(main_valleys) - 1):
        rep_start = main_valleys[i]
        rep_end = main_valleys[i+1]
        
        # Draw Main Rep Boundary
        fig.add_vline(x=df.iloc[rep_start]['Timestamp'], line_width=3, line_color="green")
        
        # Isolate the window
        rep_df = df.iloc[rep_start:rep_end]
        
        # Find ALL valleys within this window
        # We find valleys, then pick the 3 that are the 'lowest' (most stable)
        all_valleys, properties = find_peaks(-rep_df['moving_avg'], distance=100)
        
        # Sort valleys by their height (lowest value first) and take the top 3
        # properties['peak_heights'] gives the magnitude; we use negative because we inverted the signal
        if len(all_valleys) >= 2:
            # Sort by the actual signal value
            sorted_valleys = sorted(all_valleys, key=lambda v: rep_df['moving_avg'].iloc[v])
            top_3_valleys = sorted(sorted_valleys[:2]) # Sorted back by timestamp order
            
            # Draw the 3 lines
            for sv in top_3_valleys:
                stage_ts = rep_df.iloc[sv]['Timestamp']
                fig.add_vline(x=stage_ts, line_width=2, line_dash="dash", line_color="yellow")

    fig.show(renderer="colab+vscode")

In [ ]:
# Usage loop:
for i in [f"experiments_0004",f"experiments_0006"]:
    exp_id = i
    print(f"Processing {exp_id}...")
    df_ = store.get_entry_raw_data("experiments", exp_id)
    # You can change the window size here if needed (e.g., segmentation(df_, window=20))
    segmentation(df_)

Processing experiments_0004...


Processing experiments_0006...


In [ ]:
# Usage loop:
for i in [f"experiments_0001",f"experiments_0002"]:
    exp_id = i
    print(f"Processing {exp_id}...")
    df_ = store.get_entry_raw_data("experiments", exp_id)
    # You can change the window size here if needed (e.g., segmentation(df_, window=20))
    segmentation(df_,config=('device10_IMU1'), window=50)
